In [33]:
import pandas as pd

In [34]:
url = "https://docs.google.com/spreadsheets/d/1XafG9cF_xyacJbUsgXGx5an5v3uhMUHUNDDXVLtCscA/edit?usp=sharing"

In [35]:
sheet_id = url.split("/d/")[1].split("/")[0]

csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

In [36]:
df = pd.read_csv(csv_url)

In [37]:
df.head()

,Processo,Etapa,Origem,Parâmetro avaliado,Referência,Corresponde à referência?
0,1. SEI_60.016754_2026_14,Dados extraídos,Documento principal,Razão Social,SOUZA COSMÉTICOS LTDA,SIM
1,1. SEI_60.016754_2026_14,Dados extraídos,Documento principal,CNPJ,48.549.664/0001-30,SIM
2,1. SEI_60.016754_2026_14,Dados extraídos,Documento principal,Endereço,"R: Olympio Lopes dos Santos, Nº 231, São Loure...",SIM
3,1. SEI_60.016754_2026_14,Dados extraídos,Documento principal,Número do Processo,60.016754/2026-14,SIM
4,1. SEI_60.016754_2026_14,Checklist Documental,Documento principal,Tipo,Inicial,SIM


In [38]:
df.columns

Index(['Processo', 'Etapa', 'Origem', 'Parâmetro avaliado', 'Referência',
       'Corresponde à referência?'],
      dtype='str')

In [39]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 243 entries, 0 to 242
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   Processo                   243 non-null    str  
 1   Etapa                      243 non-null    str  
 2   Origem                     243 non-null    str  
 3   Parâmetro avaliado         243 non-null    str  
 4   Referência                 225 non-null    str  
 5   Corresponde à referência?  225 non-null    str  
dtypes: str(6)
memory usage: 38.6 KB


In [40]:
# Valores preenchidos e vazios por coluna
analise = pd.DataFrame({
    "Preenchidos": df.notna().sum(),
    "Vazios": df.isna().sum()
})

analise

,Preenchidos,Vazios
Processo,243,0
Etapa,243,0
Origem,243,0
Parâmetro avaliado,243,0
Referência,225,18
Corresponde à referência?,225,18


In [41]:
df = df.dropna()

In [42]:
df.info()

<class 'pandas.DataFrame'>
Index: 225 entries, 0 to 242
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   Processo                   225 non-null    str  
 1   Etapa                      225 non-null    str  
 2   Origem                     225 non-null    str  
 3   Parâmetro avaliado         225 non-null    str  
 4   Referência                 225 non-null    str  
 5   Corresponde à referência?  225 non-null    str  
dtypes: str(6)
memory usage: 38.2 KB


In [43]:
# Avaliação geral da ferramenta

df["Acerto"] = (
    df["Corresponde à referência?"]
    .str.strip()
    .str.upper()
    .eq("SIM")
)

total = len(df)
acertos = df["Acerto"].sum()
erros = total - acertos
percentual_acerto = acertos / total

print("AVALIAÇÃO GERAL DA FERRAMENTA")
print("-" * 40)
print(f"Total de conferências avaliadas: {total}")
print(f"Acertos: {acertos}")
print(f"Erros: {erros}")
print(f"Percentual de acerto: {percentual_acerto:.2%}")

AVALIAÇÃO GERAL DA FERRAMENTA
----------------------------------------
Total de conferências avaliadas: 225
Acertos: 209
Erros: 16
Percentual de acerto: 92.89%


In [44]:
resultado_etapa = (
    df.groupby("Etapa")["Acerto"]
    .agg(
        Total="count",
        Acertos="sum"
    )
    .reset_index()
)

resultado_etapa["Erros"] = (
    resultado_etapa["Total"] - resultado_etapa["Acertos"]
)

resultado_etapa["Percentual de acerto"] = (
    resultado_etapa["Acertos"] / resultado_etapa["Total"]
)

resultado_etapa.sort_values(
    "Percentual de acerto",
    ascending=False
)

,Etapa,Total,Acertos,Erros,Percentual de acerto
2,Dados extraídos,108,106,2,0.981481
0,CNAE e Risco,27,24,3,0.888889
1,Checklist Documental,90,79,11,0.877778


In [45]:
resultado_origem = (
    df.groupby("Origem")["Acerto"]
    .agg(
        Total="count",
        Acertos="sum"
    )
    .reset_index()
)

resultado_origem["Erros"] = (
    resultado_origem["Total"] - resultado_origem["Acertos"]
)

resultado_origem["Percentual de acerto"] = (
    resultado_origem["Acertos"] / resultado_origem["Total"]
)

resultado_origem.sort_values(
    "Percentual de acerto",
    ascending=False
)

,Origem,Total,Acertos,Erros,Percentual de acerto
1,Documento principal,135,133,2,0.985185
2,Tabela CNAE,27,24,3,0.888889
0,CNPJ,63,52,11,0.825397


In [46]:
resultado_parametro = (
    df.groupby("Parâmetro avaliado")["Acerto"]
    .agg(
        Total="count",
        Acertos="sum"
    )
    .reset_index()
)

resultado_parametro["Erros"] = (
    resultado_parametro["Total"] - resultado_parametro["Acertos"]
)

resultado_parametro["Percentual de acerto"] = (
    resultado_parametro["Acertos"] / resultado_parametro["Total"]
)

resultado_parametro.sort_values(
    "Percentual de acerto",
    ascending=False
)

,Parâmetro avaliado,Total,Acertos,Erros,Percentual de acerto
3,Endereço,27,27,0,1.000000
8,Tipo,27,27,0,1.000000
5,Número do Processo,27,27,0,1.000000
6,Razão Social,27,26,1,0.962963
2,CNPJ,27,26,1,0.962963
4,Grau de Risco,27,24,3,0.888889
1,CNAE secundários,21,18,3,0.857143
0,CNAE principal,21,17,4,0.809524
7,Situação cadastral,21,17,4,0.809524
